# AI Programming — Lecture 21
## Lab 3: Florence-2 Multitask Vision-Language Model

Florence-2는 하나의 pretrained model에서
**task prompt만 바꾸어 여러 vision-language task**를 수행할 수 있습니다.

이번 실습에서는 다음 세 가지를 확인합니다.

1. **Image Captioning**
2. **Object Detection**
3. **Phrase Grounding**

### 핵심 아이디어

```text
Image + Task Prompt
        ↓
    Florence-2
        ↓
Text / Label / Location
```

### 학습 목표

- 하나의 pretrained model을 여러 task에 재사용할 수 있습니다.
- Task prompt가 model의 수행 작업을 결정한다는 점을 이해합니다.
- Captioning output과 localization output의 차이를 확인합니다.
- Bounding box를 시각화할 수 있습니다.
- Text phrase를 이미지의 위치와 연결하는 grounding을 이해합니다.

> 이번 실습에서도 별도의 fine-tuning은 수행하지 않습니다.

## 1. 필요한 패키지 설치

현재 Transformers의 Florence-2 지원을 사용합니다.
별도의 특정 구버전으로 고정하지 않습니다.

In [ ]:
!pip install -q -U transformers accelerate pillow requests matplotlib

## 2. 라이브러리와 실행 환경

In [ ]:
import requests

from PIL import Image

import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import transformers

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
)

print("PyTorch version     :", torch.__version__)
print("Transformers version:", transformers.__version__)
print("CUDA available      :", torch.cuda.is_available())

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

## 3. 이미지 불러오기

In [ ]:
image_url = (
    "https://storage.googleapis.com/"
    "sfr-vision-language-research/"
    "BLIP/demo.jpg"
)

image = Image.open(
    requests.get(
        image_url,
        stream=True
    ).raw
).convert("RGB")

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Input image")
plt.show()

## 4. Florence-2 Pretrained Model 불러오기

사용 모델:

```text
microsoft/Florence-2-base-ft
```

`base-ft`는 다양한 downstream task에 맞게 fine-tuning된 checkpoint입니다.

In [ ]:
model_name = (
    "microsoft/Florence-2-base-ft"
)

processor = (
    AutoProcessor.from_pretrained(
        model_name,
        trust_remote_code=True,
    )
)

model = (
    AutoModelForMultimodalLM
    .from_pretrained(
        model_name,
        trust_remote_code=True,
        device_map="auto",
    )
)

model.eval()

print("Loaded model:", model_name)
print("Model device:", model.device)

## 5. Florence-2 실행 함수

Florence-2는 task를 special prompt로 지정합니다.

예:

```text
<CAPTION>
<OD>
<CAPTION_TO_PHRASE_GROUNDING>
```

Model이 생성한 raw token sequence는
`post_process_generation()`을 이용해 task에 맞는 구조로 변환합니다.

In [ ]:
def run_florence(
    task_prompt,
    image,
    text_input=None,
    max_new_tokens=128,
):
    if text_input is None:
        prompt = task_prompt
    else:
        prompt = (
            task_prompt
            + text_input
        )

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=3,
        )

    generated_text = (
        processor.batch_decode(
            generated_ids,
            skip_special_tokens=False,
        )[0]
    )

    parsed_answer = (
        processor.post_process_generation(
            generated_text,
            task=task_prompt,
            image_size=(
                image.width,
                image.height,
            ),
        )
    )

    return (
        generated_text,
        parsed_answer,
    )

# Part I. Image Captioning

## 6. `<CAPTION>`

이미지를 설명하는 text sequence를 생성합니다.

In [ ]:
caption_raw, caption_result = (
    run_florence(
        "<CAPTION>",
        image,
        max_new_tokens=50,
    )
)

print("Raw model output:")
print(caption_raw)

print("\nParsed result:")
print(caption_result)

# Part II. Object Detection

## 7. `<OD>`

이미지 안의 object를 찾고
label과 bounding box를 생성합니다.

In [ ]:
od_raw, od_result = run_florence(
    "<OD>",
    image,
    max_new_tokens=256,
)

print("Raw model output:")
print(od_raw)

print("\nParsed result:")
print(od_result)

## 8. Detection 결과 시각화

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 8)
)

ax.imshow(image)
ax.axis("off")

bboxes = od_result["<OD>"]["bboxes"]
labels = od_result["<OD>"]["labels"]

for bbox, label in zip(
    bboxes,
    labels,
):
    x1, y1, x2, y2 = bbox

    w = x2 - x1
    h = y2 - y1

    rect = patches.Rectangle(
        (x1, y1),
        w,
        h,
        linewidth=2,
        edgecolor="red",
        facecolor="none",
    )

    ax.add_patch(rect)

    ax.text(
        x1,
        max(y1 - 5, 0),
        label,
        fontsize=10,
        bbox=dict(
            facecolor="yellow",
            alpha=0.7,
        ),
    )

plt.title(
    "Florence-2 Object Detection"
)
plt.show()

# Part III. Phrase Grounding

## 9. Text Phrase를 이미지 위치에 연결하기

Grounding은 단순히 이미지 전체와 text가 관련 있는지를 판단하는 것이 아니라,
**특정 text phrase가 이미지의 어느 위치를 의미하는지** 찾습니다.

Task:

```text
<CAPTION_TO_PHRASE_GROUNDING>
```

In [ ]:
phrase = "dog"

ground_raw, ground_result = (
    run_florence(
        "<CAPTION_TO_PHRASE_GROUNDING>",
        image,
        text_input=phrase,
        max_new_tokens=128,
    )
)

print("Phrase:", phrase)

print("\nRaw model output:")
print(ground_raw)

print("\nParsed result:")
print(ground_result)

## 10. Grounding 결과 시각화

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 8)
)

ax.imshow(image)
ax.axis("off")

result = ground_result[
    "<CAPTION_TO_PHRASE_GROUNDING>"
]

bboxes = result["bboxes"]
labels = result["labels"]

for bbox, label in zip(
    bboxes,
    labels,
):
    x1, y1, x2, y2 = bbox

    w = x2 - x1
    h = y2 - y1

    rect = patches.Rectangle(
        (x1, y1),
        w,
        h,
        linewidth=2,
        edgecolor="blue",
        facecolor="none",
    )

    ax.add_patch(rect)

    ax.text(
        x1,
        max(y1 - 5, 0),
        label,
        fontsize=10,
        bbox=dict(
            facecolor="cyan",
            alpha=0.7,
        ),
    )

plt.title(
    f"Phrase Grounding: {phrase}"
)

plt.show()

## 11. 직접 해보기

### Caption

다음 task prompt도 선택적으로 확인할 수 있습니다.

```text
<DETAILED_CAPTION>
<MORE_DETAILED_CAPTION>
```

### Grounding

`phrase`를 바꾸어 보세요.

```text
dog
person
woman
grass
```

### 생각해 보기

1. Captioning과 Object Detection의 output은 무엇이 다른가요?
2. Object Detection과 Phrase Grounding은 무엇이 다른가요?
3. Florence-2는 왜 여러 vision task를 하나의 sequence generation 문제로 표현할 수 있을까요?
4. Grounding 정보가 robot이나 embodied AI에서 어떤 방식으로 활용될 수 있을까요?

# 정리

Lecture 21의 세 실습을 연결하면 다음과 같습니다.

```text
CLIP / SigLIP
→ Image-Text Alignment
→ Zero-Shot Classification

BLIP
→ Image-Grounded Language Generation
→ Caption / VQA

Florence-2
→ Unified Multitask Vision-Language Model
→ Caption / Detection / Grounding
```

### 꼭 기억할 것

1. Pretrained VLM은 별도의 학습 없이도 다양한 task를 수행할 수 있습니다.
2. CLIP/SigLIP은 image-text alignment를 직접 경험하기 좋습니다.
3. BLIP은 vision 정보를 바탕으로 language를 생성합니다.
4. Florence-2는 task prompt를 이용해 generation과 localization을 하나의 interface로 다룹니다.